In [1]:
import torch
import torch.nn as nn

from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

encoder = mobilenet_v3_small(
    weights=MobileNet_V3_Small_Weights.DEFAULT
)

print(encoder)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\User/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:18<00:00, 567kB/s]


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16

In [2]:
for index, layer in enumerate(encoder.features):

    print(
        f"Layer {index}:",
        type(layer).__name__
    )

Layer 0: Conv2dNormActivation
Layer 1: InvertedResidual
Layer 2: InvertedResidual
Layer 3: InvertedResidual
Layer 4: InvertedResidual
Layer 5: InvertedResidual
Layer 6: InvertedResidual
Layer 7: InvertedResidual
Layer 8: InvertedResidual
Layer 9: InvertedResidual
Layer 10: InvertedResidual
Layer 11: InvertedResidual
Layer 12: Conv2dNormActivation


In [3]:
encoder.eval()

x = torch.randn(1, 3, 512, 512)

print("Input:", x.shape)

with torch.inference_mode():

    for index, layer in enumerate(encoder.features):

        x = layer(x)

        print(
            f"Layer {index:02d}:",
            tuple(x.shape)
        )

Input: torch.Size([1, 3, 512, 512])
Layer 00: (1, 16, 256, 256)
Layer 01: (1, 16, 128, 128)
Layer 02: (1, 24, 64, 64)
Layer 03: (1, 24, 64, 64)
Layer 04: (1, 40, 32, 32)
Layer 05: (1, 40, 32, 32)
Layer 06: (1, 40, 32, 32)
Layer 07: (1, 48, 32, 32)
Layer 08: (1, 48, 32, 32)
Layer 09: (1, 96, 16, 16)
Layer 10: (1, 96, 16, 16)
Layer 11: (1, 96, 16, 16)
Layer 12: (1, 576, 16, 16)


In [4]:
original_conv = encoder.features[0][0]

original_weights = (
    original_conv.weight.detach().clone()
)

print(
    "Original weights:",
    original_weights.shape
)

new_conv = nn.Conv2d(
    in_channels=2,
    out_channels=16,
    kernel_size=3,
    stride=2,
    padding=1,
    bias=False
)

with torch.no_grad():

    adapted_weights = (
        original_weights
        .mean(dim=1, keepdim=True)
        .repeat(1, 2, 1, 1)
        * (3 / 2)
    )

    new_conv.weight.copy_(adapted_weights)

encoder.features[0][0] = new_conv

print(
    "Adapted weights:",
    encoder.features[0][0].weight.shape
)

print("SAR channel adaptation successful!")

Original weights: torch.Size([16, 3, 3, 3])
Adapted weights: torch.Size([16, 2, 3, 3])
SAR channel adaptation successful!


In [5]:
encoder.eval()

x = torch.randn(1, 2, 512, 512)

selected_layers = [0, 1, 3, 8, 12]

feature_maps = {}

with torch.inference_mode():

    for index, layer in enumerate(
        encoder.features
    ):

        x = layer(x)

        if index in selected_layers:

            feature_maps[index] = x

            print(
                f"Layer {index:02d}:",
                tuple(x.shape)
            )

Layer 00: (1, 16, 256, 256)
Layer 01: (1, 16, 128, 128)
Layer 03: (1, 24, 64, 64)
Layer 08: (1, 48, 32, 32)
Layer 12: (1, 576, 16, 16)


In [6]:
from floodvision.models.blocks import (
    DepthwiseSeparableConv,
    LightweightDecoderBlock
)

standard_conv = nn.Conv2d(
    64, 128,
    kernel_size=3,
    padding=1,
    bias=False
)

separable_conv = DepthwiseSeparableConv(
    64, 128
)

standard_params = sum(
    p.numel()
    for p in standard_conv.parameters()
)

separable_params = sum(
    p.numel()
    for p in separable_conv.parameters()
)

print("Standard Conv:", standard_params)

print("Separable Conv Block:", separable_params)

print(
    "Reduction:",
    f"{(1 - separable_params / standard_params) * 100:.2f}%"
)

Standard Conv: 73728
Separable Conv Block: 9152
Reduction: 87.59%


In [7]:
decoder = LightweightDecoderBlock(
    in_channels=128,
    skip_channels=48,
    out_channels=64
)

decoder.eval()

x = torch.randn(1, 128, 16, 16)

skip = torch.randn(1, 48, 32, 32)

with torch.inference_mode():

    output = decoder(x, skip)

print("Input:", x.shape)
print("Skip:", skip.shape)
print("Output:", output.shape)

assert output.shape == (1, 64, 32, 32)

assert torch.isfinite(output).all()

print("Lightweight decoder validation successful!")

Input: torch.Size([1, 128, 16, 16])
Skip: torch.Size([1, 48, 32, 32])
Output: torch.Size([1, 64, 32, 32])
Lightweight decoder validation successful!


In [8]:
from floodvision.models.mobilenet_unet import (
    MobileNetUNet
)

model = MobileNetUNet(
    in_channels=2,
    out_channels=1,
    pretrained=True
)

print(model)

MobileNetUNet(
  (encoder0): Conv2dNormActivation(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): Hardswish()
  )
  (encoder1): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (1): SqueezeExcitation(
        (avgpool): AdaptiveAvgPool2d(output_size=1)
        (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
        (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
        (activation): ReLU()
        (scale_activation): Hardsigmoid()
      )
      (2): Conv2dNormActivation(
        (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
   

In [9]:
from floodvision.models.baseline_cnn import (
    BaselineCNN
)

from floodvision.models.unet import UNet

from floodvision.models.resnet_unet import (
    ResNetUNet
)

models = {
    "Baseline CNN": BaselineCNN(),
    "U-Net": UNet(),
    "ResNet18 U-Net": ResNetUNet(
        pretrained=False
    ),
    "MobileNetV3 U-Net": model
}

for name, network in models.items():

    total_params = sum(
        p.numel()
        for p in network.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in network.parameters()
        if p.requires_grad
    )

    size_mib = (
        total_params * 4 / 1024**2
    )

    print(f"\n{name}")

    print(
        f"Total parameters: {total_params:,}"
    )

    print(
        f"Trainable parameters: {trainable_params:,}"
    )

    print(
        f"FP32 weights: {size_mib:.2f} MiB"
    )


Baseline CNN
Total parameters: 434,209
Trainable parameters: 434,209
FP32 weights: 1.66 MiB

U-Net
Total parameters: 482,593
Trainable parameters: 482,593
FP32 weights: 1.84 MiB

ResNet18 U-Net
Total parameters: 14,238,145
Trainable parameters: 14,238,145
FP32 weights: 54.31 MiB

MobileNetV3 U-Net
Total parameters: 1,034,057
Trainable parameters: 1,034,057
FP32 weights: 3.94 MiB


In [10]:
model.eval()

x = torch.randn(
    1, 2, 512, 512
)

with torch.inference_mode():

    output = model(x)

print("Input shape:", x.shape)

print("Output shape:", output.shape)

print("Output dtype:", output.dtype)

print("Minimum logit:", output.min().item())

print("Maximum logit:", output.max().item())

assert output.shape == (1, 1, 512, 512)

assert torch.isfinite(output).all()

print("\nForward pass successful!")

Input shape: torch.Size([1, 2, 512, 512])
Output shape: torch.Size([1, 1, 512, 512])
Output dtype: torch.float32
Minimum logit: 0.016871042549610138
Maximum logit: 0.049688681960105896

Forward pass successful!


In [11]:
from floodvision.losses.combined import (
    BCEDiceLoss
)

model.train()

model.zero_grad(set_to_none=True)

x = torch.randn(
    2, 2, 128, 128
)

target = torch.randint(
    0,
    2,
    (2, 1, 128, 128)
).float()

# Simulate invalid SAR pixels
target[:, :, :10, :10] = -1

criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5,
    pos_weight=None
)

logits = model(x)

loss = criterion(logits, target)

assert torch.isfinite(loss)

loss.backward()

print("Loss:", loss.item())

print("Backward pass successful!")

Loss: 0.6147519946098328
Backward pass successful!


In [12]:
missing_gradients = []

invalid_gradients = []

nonzero_gradients = 0

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.grad is None:

        missing_gradients.append(name)

        continue

    if not torch.isfinite(param.grad).all():

        invalid_gradients.append(name)

    if torch.count_nonzero(param.grad).item() > 0:

        nonzero_gradients += 1

print("Missing gradients:", missing_gradients)

print("Invalid gradients:", invalid_gradients)

print(
    "Non-zero gradient tensors:",
    nonzero_gradients
)

assert not missing_gradients

assert not invalid_gradients

assert nonzero_gradients > 0

print("\nGradient validation successful!")

Missing gradients: []
Invalid gradients: []
Non-zero gradient tensors: 203

Gradient validation successful!


In [13]:
encoder_grad = (
    model.encoder0[0]
    .weight.grad.norm().item()
)

decoder_grad = (
    model.decoder3.projection[0]
    .weight.grad.norm().item()
)

output_grad = (
    model.output.weight
    .grad.norm().item()
)

print("Encoder gradient:", encoder_grad)

print("Decoder gradient:", decoder_grad)

print("Output gradient:", output_grad)

assert encoder_grad > 0

assert decoder_grad > 0

assert output_grad > 0

print("\nEnd-to-end gradient flow verified!")

Encoder gradient: 1.0245883464813232
Decoder gradient: 0.1600799262523651
Output gradient: 0.15358366072177887

End-to-end gradient flow verified!


In [14]:
encoder_grad = model.encoder0[0].weight.grad.norm().item()

decoder_grad = (
    model.decoder3.projection[0]
    .weight.grad.norm().item()
)

output_grad = model.output.weight.grad.norm().item()

print("Encoder gradient:", encoder_grad)
print("Decoder gradient:", decoder_grad)
print("Output gradient:", output_grad)

assert encoder_grad > 0
assert decoder_grad > 0
assert output_grad > 0

print("End-to-end gradient flow verified!")

Encoder gradient: 1.0245883464813232
Decoder gradient: 0.1600799262523651
Output gradient: 0.15358366072177887
End-to-end gradient flow verified!


In [15]:
import importlib
import floodvision.models.factory as factory

importlib.reload(factory)

config = {
    "name": "mobilenet_unet",
    "in_channels": 2,
    "out_channels": 1,
    "pretrained": False
}

factory_model = factory.build_model(config)

params = sum(
    p.numel()
    for p in factory_model.parameters()
)

print("Available models:")
print(list(factory.MODEL_REGISTRY.keys()))

print("Model:", type(factory_model).__name__)

print(f"Parameters: {params:,}")

assert params == 1034057

print("Model factory validation successful!")

Available models:
['baseline_cnn', 'unet', 'resnet_unet', 'mobilenet_unet']
Model: MobileNetUNet
Parameters: 1,034,057
Model factory validation successful!
